In [ ]:
# ============================================
# CELL 1: Import libraries and load data
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visual style
sns.set_theme(style="whitegrid")
%matplotlib inline

# Load the dataset
# encoding='latin1' handles special characters
df = pd.read_csv('../data/Amazon.csv', encoding='latin1')

print("✅ Data Loaded Successfully!")
print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n📋 Columns:\n{list(df.columns)}")

In [ ]:
# ============================================
# CELL 2: First look at raw data
# ============================================
# Always inspect data BEFORE cleaning
# This tells us what problems exist

print("🔍 FIRST 5 ROWS:")
print("="*60)
display(df.head())

print("\n📋 DATA TYPES:")
print("="*60)
print(df.dtypes)

print("\n❓ MISSING VALUES:")
print("="*60)
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum()/len(df)*100).round(2)
})
print(missing[missing['Missing Count'] > 0]
      .sort_values('Missing %', ascending=False))

print(f"\n🔁 DUPLICATE ROWS: {df.duplicated().sum()}")

In [ ]:
# ============================================
# CELL 3: Clean column names
# ============================================
# Problem: Column names may have spaces,
# special characters, or inconsistent cases
# Solution: Make them clean and uniform

print("BEFORE cleaning column names:")
print(list(df.columns))

# Remove spaces, lowercase everything,
# replace hyphens and spaces with underscore
df.columns = (df.columns
              .str.strip()        # remove leading/trailing spaces
              .str.lower()        # make lowercase
              .str.replace(' ', '_')   # spaces to underscore
              .str.replace('-', '_'))  # hyphens to underscore

print("\nAFTER cleaning column names:")
print(list(df.columns))
print("\n✅ Column names cleaned!")

In [ ]:
# ============================================
# CELL 4: Remove duplicate rows
# ============================================
# Duplicate rows = same order recorded twice
# This inflates revenue numbers — very dangerous!

before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"📊 Rows BEFORE removing duplicates: {before}")
print(f"📊 Rows AFTER removing duplicates:  {after}")
print(f"🗑️  Duplicates removed: {before - after}")
print("\n✅ Duplicates removed!")

In [ ]:
# ============================================
# CELL 5: Fix the Date column
# ============================================
# Problem: Date column is stored as TEXT (object)
# Solution: Convert to proper DateTime format
# Why: We need to extract Month, Year for analysis

# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'], 
                             infer_datetime_format=True, 
                             errors='coerce')

# Extract useful time features
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%B')  # January, February...
df['day_of_week'] = df['date'].dt.strftime('%A') # Monday, Tuesday...
df['quarter'] = df['date'].dt.quarter            # Q1, Q2, Q3, Q4

print("✅ Date column fixed!")
print(f"\n📅 Date Range:")
print(f"   From: {df['date'].min()}")
print(f"   To:   {df['date'].max()}")
print(f"\n🗓️  Unique Months: {df['month_name'].nunique()}")
print(f"\nNew columns added: year, month, month_name, day_of_week, quarter")

In [ ]:
# ============================================
# CELL 6: Handle missing values
# ============================================
# Strategy:
#   Numeric columns  → fill with median
#   Text columns     → fill with 'Unknown'
#   Critical columns → drop rows

print("Missing values BEFORE cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill numeric columns with median
# Median is better than mean for skewed data
numeric_cols = df.select_dtypes(include=['number']).columns
for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    
# Fill text columns with 'Unknown'
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
    df[col] = df[col].fillna('Unknown')

print("\nMissing values AFTER cleaning:")
remaining = df.isnull().sum().sum()
print(f"Total missing values remaining: {remaining}")
print("\n✅ Missing values handled!")

In [ ]:
# ============================================
# CELL 7: Fix data types
# ============================================
# qty = quantity (must be integer)
# amount = money (must be float)
# Remove rows where qty or amount = 0 or negative

# Convert qty to numeric
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

# Drop rows where amount or qty is 0 or negative
# Why: Zero amount orders are cancelled/invalid
before = len(df)
df = df[df['amount'] > 0]
df = df[df['qty'] > 0]
after = len(df)

print(f"Rows removed (zero/negative values): {before - after}")

# Reset index after all cleaning
df = df.reset_index(drop=True)

print(f"\n✅ Final clean dataset shape: {df.shape}")
print(f"📊 Rows: {df.shape[0]}")
print(f"📋 Columns: {df.shape[1]}")

In [ ]:
# ============================================
# CELL 8: Add business columns
# ============================================
# These new columns help us answer
# real business questions

# Total Revenue per row
df['revenue'] = df['qty'] * df['amount']

# Order Size Category
def categorize_order(amount):
    if amount < 500:
        return 'Small Order'
    elif amount < 2000:
        return 'Medium Order'
    else:
        return 'Large Order'

df['order_size'] = df['amount'].apply(categorize_order)

print("✅ Business columns added!")
print("\nNew columns:")
print("  → revenue    : qty × amount")
print("  → order_size : Small / Medium / Large")
print(f"\nRevenue Summary:")
print(f"  Total Revenue : ₹{df['revenue'].sum():,.2f}")
print(f"  Avg Order Value: ₹{df['amount'].mean():,.2f}")
print(f"  Max Order     : ₹{df['amount'].max():,.2f}")

In [ ]:
# ============================================
# CELL 9: Save the cleaned dataset
# ============================================
# Always save cleaned data separately
# NEVER overwrite your original raw data

clean_path = '../data/Amazon_cleaned.csv'
df.to_csv(clean_path, index=False)

print(f"✅ Clean data saved to: {clean_path}")
print(f"\n📊 Final Dataset Summary:")
print(f"   Total Orders  : {len(df):,}")
print(f"   Total Revenue : ₹{df['revenue'].sum():,.2f}")
print(f"   Date Range    : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"   Categories    : {df['category'].nunique()}")
print(f"   Cities        : {df['ship_city'].nunique()}")